# Correlation and the slope

The sample correlation measures linear association on a scale with no units. The regression slope measures the same association in the units of $y$ per unit of $x$. They are two normalizations of one sum of products, and they answer different questions.


## The identity

Define the unnormalized sums of squares and products by

$$
S_{xx} = \sum (x_i-\bar x)^2, \qquad
S_{yy} = \sum (y_i-\bar y)^2, \qquad
S_{xy} = \sum (x_i-\bar x)(y_i-\bar y).
$$

The correlation and the slope are

$$
r = \dfrac{S_{xy}}{\sqrt{S_{xx}S_{yy}}}, \qquad b_1 = \dfrac{S_{xy}}{S_{xx}}.
$$

Eliminate $S_{xy}$:

$$
b_1 = r\sqrt{\dfrac{S_{yy}}{S_{xx}}} = r\,\dfrac{s_y}{s_x},
$$

where $s_x$ and $s_y$ are the sample standard deviations. The divisor $n-1$ appears in both and cancels in the ratio. The sign of $b_1$ is the sign of $r$. Squaring the correlation recovers $R^2$ for a simple regression with an intercept:

$$
r^2 = \dfrac{S_{xy}^2}{S_{xx}S_{yy}} = \dfrac{b_1^2 S_{xx}}{S_{yy}} = \dfrac{\mathrm{SSR}}{\mathrm{TSS}} = R^2,
$$

because $\mathrm{SSR} = b_1^2 S_{xx}$ once the fitted values are $b_0 + b_1 x$ and $b_0 = \bar y - b_1\bar x$.


## The five-point numbers

The centered sums were $S_{xx} = 10$, $S_{xy} = 20$, and $S_{yy} = \mathrm{TSS} = 44$. Thus

$$
r = \dfrac{20}{\sqrt{10\cdot 44}} = \dfrac{20}{\sqrt{440}} = \dfrac{20}{2\sqrt{110}} = \sqrt{\dfrac{10}{11}},
$$

taking the positive root because $S_{xy} > 0$. Then $r^2 = 10/11$, which is the $R^2$ already computed.

The sample standard deviations use $n - 1 = 4$:

$$
s_x = \sqrt{\dfrac{10}{4}} = \sqrt{\dfrac{5}{2}}, \qquad s_y = \sqrt{\dfrac{44}{4}} = \sqrt{11}.
$$

The product formula returns the slope:

$$
r\,\dfrac{s_y}{s_x}
= \sqrt{\dfrac{10}{11}}\sqrt{\dfrac{11}{5/2}}
= \sqrt{\dfrac{10}{11}\cdot\dfrac{22}{5}}
= \sqrt{4} = 2 = b_1.
$$


In [1]:
# Correlation, its square, and the conversion from correlation to slope.
from fractions import Fraction
import math
sxx, syy, sxy = Fraction(10), Fraction(44), Fraction(20)
r_squared = (sxy ** 2) / (sxx * syy)
sx = math.sqrt(float(sxx / 4))
sy = math.sqrt(float(syy / 4))
r = math.sqrt(float(r_squared))
print("r squared", r_squared)
print("r * sy / sx", r * sy / sx)
assert r_squared == Fraction(10, 11)
assert abs(r * sy / sx - 2) < 1e-12


r squared 10/11
r * sy / sx 2.0


## A perfect line and a sign change

On $(x, y) = (1, 3),\ (2, 5),\ (3, 7)$, the relation is $y = 1 + 2x$ with no error. Then $r = 1$ and $b_1 = 2$.

Replace $y$ by $-y$. Every product in $S_{xy}$ changes sign, so $r$ changes sign and the slope changes sign. $R^2 = r^2$ does not change. Correlation keeps a direction; $R^2$ does not.


In [2]:
# Negating y negates the correlation and leaves R-squared unchanged.
from fractions import Fraction
x = [1, 2, 3]
y = [3, 5, 7]

def correlation(predictors, responses):
    n = len(predictors)
    xb = Fraction(sum(predictors), n)
    yb = Fraction(sum(responses), n)
    sxx = sum((a - xb) ** 2 for a in predictors)
    syy = sum((b - yb) ** 2 for b in responses)
    sxy = sum((a - xb) * (b - yb) for a, b in zip(predictors, responses))
    return sxy, sxy ** 2 / (sxx * syy)

products, r2 = correlation(x, y)
products_flipped, r2_flipped = correlation(x, [-yi for yi in y])
print("Sxy and r2", products, r2)
print("after negating y", products_flipped, r2_flipped)
assert products == 4 and r2 == 1
assert products_flipped == -products and r2_flipped == r2


Sxy and r2 4 1
after negating y -4 1


## Pitfall: no linear association is not no association

Let $x = -2, -1, 0, 1, 2$ and $y = x^2$, so $y = 4, 1, 0, 1, 4$. Then $\bar x = 0$ and

$$
S_{xy} = \sum x_i y_i = (-2)(4) + (-1)(1) + 0 + (1)(1) + (2)(4) = 0.
$$

Hence $r = 0$ and the least-squares slope is $0$. The best line is the horizontal line at $\bar y = 2$. The points nevertheless lie on the parabola $y = x^2$, which is a deterministic relationship. Correlation and simple linear regression are silent about every pattern that is not a straight line.

A second misreading is causal. If a count of cups of tea and a count of pages read both increase exactly with the day index $1, 2, 3, 4$, their correlation is $1$. The arithmetic does not say that tea produces pages. A third variable, the day, is enough to create the line.


In [3]:
# A parabola can have correlation zero. A shared trend can have correlation one.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
y = [xi ** 2 for xi in x]
sxy = sum(xi * (yi - Fraction(sum(y), 5)) for xi, yi in zip(x, y))
print("parabola Sxy", sxy, "mean response", Fraction(sum(y), 5))
day = [1, 2, 3, 4]
tea = day[:]
pages = [2 * t for t in day]

def pearson(a, b):
    n = len(a)
    ab = Fraction(sum(a), n)
    bb = Fraction(sum(b), n)
    sxx = sum((v - ab) ** 2 for v in a)
    syy = sum((v - bb) ** 2 for v in b)
    sxy_local = sum((u - ab) * (v - bb) for u, v in zip(a, b))
    return sxy_local ** 2 / (sxx * syy)

print("tea and pages, r squared", pearson(tea, pages))
assert sxy == 0
assert pearson(tea, pages) == 1


parabola Sxy 0 mean response 2
tea and pages, r squared 1


## Summary

- $b_1 = r\, s_y/s_x$. Correlation removes units; the slope puts them back.
- For simple regression with an intercept, $r^2 = R^2$. On the five-point line, $r = \sqrt{10/11}$.
- $r = 0$ does not mean the variables are unrelated, and $r = 1$ does not mean one causes the other.
